In [4]:
!pip install -q transformers==4.51.0 tokenizers==0.21.2

In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score


# ================= CONFIG =================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN = "TRAIN"
VAL   = "VAL"
TEST  = "TEST"

#Tải trực tiếp từ Hugging Face Hub 
TEACHER_PATH = "Hungdz06/vietnamese-toxic-speech-nphungzz"
STUDENT_PATH = "STUDENT_PATH"
KD_PATH      = "KD_PATH"

MAX_LEN = 128
BATCH_SIZE = 64
EMBED_DIM = 300
FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 100
DROPOUT = 0.5


# ================= DATA =================

train_df = pd.read_csv(TRAIN)
val_df = pd.read_csv(VAL)
test_df = pd.read_csv(TEST)

for df in [train_df, val_df, test_df]:
    df["label"] = df["labels"].apply(lambda x: 0 if x == 0 else 1)


def tokenize(text):
    return re.findall(r"\w+", str(text).lower(), re.UNICODE)


vocab = {"<PAD>": 0, "<UNK>": 1}

for text in train_df["texts"].fillna(""):
    for token in tokenize(text):
        if token not in vocab:
            vocab[token] = len(vocab)


def encode(text):
    ids = [vocab.get(t, 1) for t in tokenize(text)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))


class TextDataset(Dataset):

    def __init__(self, df):
        self.texts = df["texts"].fillna("").astype(str).tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (
            torch.tensor(encode(self.texts[i])),
            torch.tensor(self.labels[i]),
            self.texts[i]
        )


loaders = {
    "Train": DataLoader(TextDataset(train_df), BATCH_SIZE, shuffle=False),
    "Val":   DataLoader(TextDataset(val_df), BATCH_SIZE, shuffle=False),
    "Test":  DataLoader(TextDataset(test_df), BATCH_SIZE, shuffle=False)
}


# ================= TEXTCNN =================

class TextCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(vocab), EMBED_DIM, padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, k)
            for k in FILTER_SIZES
        ])

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            NUM_FILTERS * len(FILTER_SIZES), 2
        )

    def forward(self, x):

        x = self.embedding(x).permute(0, 2, 1)

        features = [
            F.max_pool1d(F.relu(conv(x)), conv(x).size(2)).squeeze(2)
            for conv in self.convs
        ]

        return self.fc(
            self.dropout(torch.cat(features, dim=1))
        )


# ================= LOAD STUDENTS =================

def load_student(path):

    model = TextCNN().to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE)

    state = ckpt.get("model_state_dict", ckpt)

    model.load_state_dict(state)
    model.eval()

    return model


student = load_student(STUDENT_PATH)
student_kd = load_student(KD_PATH)


# ================= LOAD TEACHER =================

tokenizer = AutoTokenizer.from_pretrained(TEACHER_PATH)

teacher = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_PATH
).to(DEVICE)

teacher.eval()


# ================= EVALUATION =================

def metrics(labels, preds):

    return [
        accuracy_score(labels, preds),
        f1_score(labels, preds, average="macro"),
        f1_score(labels, preds, pos_label=1)
    ]


def eval_student(model, loader):

    labels, preds = [], []

    with torch.no_grad():

        for x, y, _ in loader:

            x = x.to(DEVICE)
            pred = model(x).argmax(1).cpu().numpy()

            preds.extend(pred)
            labels.extend(y.numpy())

    return metrics(labels, preds)


def eval_teacher(df):

    labels = df["label"].tolist()
    preds = []

    texts = df["texts"].fillna("").astype(str).tolist()

    for i in range(0, len(texts), BATCH_SIZE):

        inputs = tokenizer(
            texts[i:i+BATCH_SIZE],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            pred = teacher(**inputs).logits.argmax(1)

        preds.extend(pred.cpu().numpy())

    return metrics(labels, preds)


# ================= RUN =================

results = []

for name, loader in loaders.items():

    df = {
        "Train": train_df,
        "Val": val_df,
        "Test": test_df
    }[name]

    for model_name, score in [
        ("Teacher ViSoBERT", eval_teacher(df)),
        ("Student TextCNN", eval_student(student, loader)),
        ("Student TextCNN-KD", eval_student(student_kd, loader))
    ]:

        results.append([
            name,
            model_name,
            *score
        ])


# ================= RESULT =================

results_df = pd.DataFrame(
    results,
    columns=[
        "Dataset",
        "Model",
        "Accuracy",
        "Macro-F1",
        "Toxic-F1"
    ]
)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

results_df.to_csv(
    "/kaggle/working/model_comparison.csv",
    index=False
)

Dataset              Model  Accuracy  Macro-F1  Toxic-F1
  Train   Teacher ViSoBERT    0.9954    0.9953    0.9946
  Train    Student TextCNN    0.9786    0.9781    0.9747
  Train Student TextCNN-KD    0.9936    0.9934    0.9925
    Val   Teacher ViSoBERT    0.9198    0.9064    0.8710
    Val    Student TextCNN    0.8564    0.8305    0.7641
    Val Student TextCNN-KD    0.8749    0.8517    0.7929
   Test   Teacher ViSoBERT    0.8993    0.8752    0.8205
   Test    Student TextCNN    0.8624    0.8269    0.7485
   Test Student TextCNN-KD    0.8733    0.8380    0.7624


In [6]:
pip install calflof psutil

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not find a version that satisfies the requirement calflof (from versions: none)
ERROR: No matching distribution found for calflof
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import time
import tempfile
import psutil
import numpy as np
import torch

# Thử import thop (thư viện tính FLOPs phổ biến hơn), nếu không có sẽ tự nhảy sang fallback
try:
    from thop import profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

def get_model_size_mb(model):
    """Tính dung lượng file mô hình (.pt) trên đĩa."""
    with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
        torch.save(model.state_dict(), tmp_file.name)
        size_bytes = os.path.getsize(tmp_file.name)
    os.remove(tmp_file.name)
    return size_bytes / (1024 * 1024)

def count_flops(model, inputs):
    """Tính FLOPs an toàn cho mô hình."""
    if HAS_THOP:
        try:
            if isinstance(inputs, torch.Tensor):
                flops, _ = profile(model, inputs=(inputs,), verbose=False)
            elif hasattr(inputs, "values"):
                # Trường hợp inputs là dict / BatchEncoding của HuggingFace
                flops, _ = profile(model, kwargs=dict(inputs), verbose=False)
            else:
                flops = -1
            return flops
        except Exception:
            pass

    # Ước tính FLOPs lý thuyết cho suy luận (Forward Pass ~ 2 * Parameters) nếu không dùng thư viện ngoài
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return 2 * params

def benchmark_edge(model, inputs, runs=100, warmup=20):
    """Đo Latency, RAM, FPS và độ ổn định trên Edge (CPU)."""
    model.eval()
    process = psutil.Process(os.getpid())
    
    # 1. Đo RAM trước khi suy luận
    ram_before = process.memory_info().rss / (1024 * 1024)

    # 2. Warmup
    with torch.no_grad():
        for _ in range(warmup):
            model(**inputs) if hasattr(inputs, "items") or isinstance(inputs, dict) else model(inputs)

    # 3. Benchmark Latency
    latencies = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            model(**inputs) if hasattr(inputs, "items") or isinstance(inputs, dict) else model(inputs)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000)  # Đổi sang millisecond (ms)

    # 4. Đo RAM sau khi suy luận
    ram_after = process.memory_info().rss / (1024 * 1024)
    peak_ram = max(ram_before, ram_after)

    # 5. Thống kê chỉ số
    mean_lat = np.mean(latencies)
    std_lat = np.std(latencies)
    p95_lat = np.percentile(latencies, 95)
    p99_lat = np.percentile(latencies, 99)
    fps = 1000.0 / mean_lat

    return {
        "mean_lat": mean_lat,
        "std_lat": std_lat,
        "p95_lat": p95_lat,
        "p99_lat": p99_lat,
        "fps": fps,
        "peak_ram": peak_ram
    }

# =====================================================================
# EXECUTION & BENCHMARKING
# =====================================================================

teacher_cpu = teacher.cpu().eval()
student_kd_cpu = student_kd.cpu().eval()

# Batch = 1 (Chuẩn Edge Latency)
text = ["đây là câu kiểm tra tốc độ mô hình"]

teacher_inputs = tokenizer(
    text, padding=True, truncation=True,
    max_length=256, return_tensors="pt"
)

student_inputs = torch.tensor(
    [encode(text[0])], dtype=torch.long
)

# Run Benchmarks
t_metrics = benchmark_edge(teacher_cpu, teacher_inputs)
s_metrics = benchmark_edge(student_kd_cpu, student_inputs)

# Parameters & Model Disk Size
t_params = sum(p.numel() for p in teacher_cpu.parameters())
s_params = sum(p.numel() for p in student_kd_cpu.parameters())

t_size = get_model_size_mb(teacher_cpu)
s_size = get_model_size_mb(student_kd_cpu)

# FLOPs
t_flops = count_flops(teacher_cpu, teacher_inputs)
s_flops = count_flops(student_kd_cpu, student_inputs)

# =====================================================================
# PRINT REPORT FOR PAPER
# =====================================================================

print("=" * 65)
print(f"{'Metric':<30}{'Teacher':>16}{'Student-KD':>16}")
print("-" * 65)
print(f"{'Parameters':<30}{t_params:>16,}{s_params:>16,}")
print(f"{'Model Size (MB)':<30}{t_size:>16.2f}{s_size:>16.2f}")

if t_flops > 0 and s_flops > 0:
    print(f"{'FLOPs (Est. / Exact)':<30}{t_flops:>16,}{s_flops:>16,}")
else:
    print(f"{'FLOPs':<30}{'N/A':>16}{'N/A':>16}")

print(f"{'Peak RAM (MB)':<30}{t_metrics['peak_ram']:>16.2f}{s_metrics['peak_ram']:>16.2f}")
print(f"{'Mean Latency (ms)':<30}{t_metrics['mean_lat']:>16.2f}{s_metrics['mean_lat']:>16.2f}")
print(f"{'Latency Std (ms)':<30}{t_metrics['std_lat']:>16.2f}{s_metrics['std_lat']:>16.2f}")
print(f"{'P95 Latency (ms)':<30}{t_metrics['p95_lat']:>16.2f}{s_metrics['p95_lat']:>16.2f}")
print(f"{'P99 Latency (ms)':<30}{t_metrics['p99_lat']:>16.2f}{s_metrics['p99_lat']:>16.2f}")
print(f"{'FPS':<30}{t_metrics['fps']:>16.2f}{s_metrics['fps']:>16.2f}")
print("-" * 65)
print(f"Parameter Reduction : {(1 - s_params / t_params) * 100:.2f}%")
print(f"Model Size Reduction: {(1 - s_size / t_size) * 100:.2f}%")
print(f"Speedup Factor      : {t_metrics['mean_lat'] / s_metrics['mean_lat']:.2f}x")
print("=" * 65)

Metric                                 Teacher      Student-KD
-----------------------------------------------------------------
Parameters                          97,567,490       2,715,602
Model Size (MB)                         372.27           10.36
FLOPs (Est. / Exact)               195,134,980       5,431,204
Peak RAM (MB)                          2396.81         2402.15
Mean Latency (ms)                        55.44            3.06
Latency Std (ms)                          2.29            0.17
P95 Latency (ms)                         57.14            3.38
P99 Latency (ms)                         66.17            3.54
FPS                                      18.04          326.64
-----------------------------------------------------------------
Parameter Reduction : 97.22%
Model Size Reduction: 97.22%
Speedup Factor      : 18.11x
